# Eval Matrix: 5 Models × 3 Tasks (E2E, DART, WebNLG)

Runs the cross-model cross-task evaluation grid in one notebook:

| Row \ Column | E2E valid | DART valid | WebNLG valid |
|---|---|---|---|
| **base GPT-2 medium** | | | |
| **base + E2E LoRA** | | | |
| **base + DART LoRA** | | | |
| **base + WebNLG LoRA** | | | |
| **base + MoLE (top-1 router)** | | | |

For each (model, task) cell:
- Generation uses the **task's** decoding settings (consistent within each column).
- Generation runs once per **unique meaning representation** (deduplicated from the raw split).
- Evaluation uses sacrebleu BLEU + NLTK METEOR + pyter3 TER applied **uniformly across all 15 cells**, so every column is apples-to-apples within itself. Absolute numbers will not match each task's paper-reported values exactly (different scorer implementations differ by ~0.5 BLEU absolute), but the within-task ranking of the 5 models is what we care about.

Recommended runtime: T4 GPU. With beam 10 (paper default) the full matrix takes ~6-8 hours; reducing to beam 4 cuts that to ~2.5 hours with marginal quality loss.

## 1. GPU + Repo + Deps

In [ ]:
!nvidia-smi

import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
from getpass import getpass
from pathlib import Path
import os, subprocess

REPO_OWNER = "justinlxiang"
REPO_NAME = "CS4782-final-project"
BRANCH = "mole-experiment"
PROJECT_DIR = Path("/content") / REPO_NAME
WORK_DIR = PROJECT_DIR / "lora-gpt2-medium-e2e"

token = getpass("GitHub token, or press Enter for public clone: ")
repo_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
if token:
    repo_url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(PROJECT_DIR)], check=True)

os.chdir(WORK_DIR)
print("cwd:", Path.cwd())
!git log --oneline -3

In [ ]:
!pip install -q -r requirements.txt
# pyter3 isn't in the pinned requirements (used only by this matrix notebook).
!pip install -q pyter3 nltk
import nltk
for r in ('wordnet', 'punkt', 'punkt_tab', 'omw-1.4'):
    try: nltk.download(r, quiet=True)
    except Exception: pass

## 2. Drive Mount

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

EVAL_OUT_LOCAL = Path('outputs/runs/eval_matrix')
EVAL_OUT_DRIVE = Path('/content/drive/MyDrive/eval_matrix')
EVAL_OUT_LOCAL.mkdir(parents=True, exist_ok=True)
EVAL_OUT_DRIVE.mkdir(parents=True, exist_ok=True)


def backup(rel_path, dest_name=None):
    src = Path(rel_path)
    if not src.exists():
        print('skip missing:', src)
        return None
    dst = EVAL_OUT_DRIVE / (dest_name or src.name)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print('backed up:', src, '->', dst)
    return dst

print('local out:', EVAL_OUT_LOCAL)
print('drive out:', EVAL_OUT_DRIVE)

## 3. Pull Adapters + Router From Drive

We need:
- Three single-task LoRA adapters: E2E, DART, WebNLG
- One trained MoLE router

These come from prior per-task / MoLE training runs, all sitting in Drive at known paths from those notebooks' final-backup cells.

In [ ]:
from pathlib import Path
import shutil

ADAPTERS = [
    ('e2e_lora_r4_alpha32',    Path('/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_final.pt')),
    ('dart_lora_r4_alpha32',   Path('/content/drive/MyDrive/dart_lora_r4_alpha32/checkpoints/adapter_final.pt')),
    ('webnlg_lora_r4_alpha32', Path('/content/drive/MyDrive/webnlg_lora_r4_alpha32/checkpoints/adapter_final.pt')),
]
ROUTER = Path('/content/drive/MyDrive/mole_e2e_dart_webnlg/checkpoints/router_final.pt')

for run, src in ADAPTERS:
    dst_dir = Path(run) / 'checkpoints'
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / 'adapter_final.pt'
    if not src.exists():
        raise FileNotFoundError(f'missing in Drive: {src}')
    shutil.copy2(src, dst)
    print(f'{run}: {dst}  ({dst.stat().st_size / 1e6:.1f} MB)')

if not ROUTER.exists():
    raise FileNotFoundError(f'missing in Drive: {ROUTER}')
ROUTER_LOCAL = Path('mole_e2e_dart_webnlg/checkpoints/router_final.pt')
ROUTER_LOCAL.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(ROUTER, ROUTER_LOCAL)
print(f'router: {ROUTER_LOCAL}  ({ROUTER_LOCAL.stat().st_size / 1e3:.1f} KB)')

## 4. Download + Prep Raw Data For All 3 Tasks

We only need the **valid** split for the eval matrix, but the prep scripts grab train/valid/test in one shot. The processed (tokenized) files are not needed for evaluation — only the raw `data/raw/<task>/valid.{txt,jsonl}` files, which the generation script reads directly via `load_e2e_records`.

In [ ]:
# 4a. E2E
!mkdir -p data/raw/e2e
!curl -L -s -o data/raw/e2e/train.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/train.txt
!curl -L -s -o data/raw/e2e/valid.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/valid.txt
!curl -L -s -o data/raw/e2e/test.txt  https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/test.txt

# 4b. DART
!mkdir -p data/raw/dart
!curl -L -s -o data/raw/dart/dart-v1.1.1-full-train.json https://raw.githubusercontent.com/Yale-LILY/dart/master/data/v1.1.1/dart-v1.1.1-full-train.json
!curl -L -s -o data/raw/dart/dart-v1.1.1-full-dev.json   https://raw.githubusercontent.com/Yale-LILY/dart/master/data/v1.1.1/dart-v1.1.1-full-dev.json
!curl -L -s -o data/raw/dart/dart-v1.1.1-full-test.json  https://raw.githubusercontent.com/Yale-LILY/dart/master/data/v1.1.1/dart-v1.1.1-full-test.json

# 4c. WebNLG
!mkdir -p data/raw/webnlg
!curl -L -s -o data/raw/webnlg/train.json https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/webnlg_challenge_2017/train.json
!curl -L -s -o data/raw/webnlg/dev.json   https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/webnlg_challenge_2017/dev.json
!curl -L -s -o data/raw/webnlg/test.json  https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/webnlg_challenge_2017/test.json

# 4d. Convert DART/WebNLG triples to {context, completion} JSONL.
!python scripts/prepare_dart_raw.py    --config configs/dart_gpt2_medium_lora.yaml
!python scripts/prepare_webnlg_raw.py  --config configs/webnlg_gpt2_medium_lora.yaml

!ls -lh data/raw/e2e/valid.txt data/raw/dart/valid.jsonl data/raw/webnlg/valid.jsonl

## 5. Configure The Run

`BEAM_OVERRIDE`: leave at `None` for paper-default (beam 10) per task. Set to e.g. `4` to cut the run from ~6-8h to ~2.5h with minor quality loss.

`MAX_EXAMPLES`: leave at `None` for the full unique-context count per task, set to e.g. `200` for a fast smoke run before committing to the full matrix.

In [ ]:
BEAM_OVERRIDE = 4      # None | 1 | 2 | 4 | 5 | 10
MAX_EXAMPLES  = 50      # None | 50 | 200 | etc

VARIANTS = [
    ('base',   None),
    ('e2e',    None),
    ('dart',   None),
    ('webnlg', None),
    ('mole',   'configs/mole_e2e_dart_webnlg.yaml'),
]
TASKS = [
    ('e2e',    'configs/e2e_gpt2_medium_lora.yaml'),
    ('dart',   'configs/dart_gpt2_medium_lora.yaml'),
    ('webnlg', 'configs/webnlg_gpt2_medium_lora.yaml'),
]
SPLIT = 'valid'
print(f'matrix: {len(VARIANTS)} variants x {len(TASKS)} tasks = {len(VARIANTS)*len(TASKS)} cells')
print(f'beam_override: {BEAM_OVERRIDE}, max_examples: {MAX_EXAMPLES}, split: {SPLIT}')

## 6. Generate Predictions (5 × 3 Cells)

Each cell calls `scripts/eval_matrix_generate.py` once. Predictions land at `outputs/runs/eval_matrix/<variant>/<task>/<split>_predictions.jsonl` for both later evaluation and reproducibility.

In [ ]:
import subprocess, time, os
from pathlib import Path
from tqdm.auto import tqdm

def gen_one(variant, mole_cfg_path, task_name, task_cfg_path):
    out_dir = EVAL_OUT_LOCAL / variant / task_name
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / f'{SPLIT}_predictions.jsonl'
    cmd = [
        'python', 'scripts/eval_matrix_generate.py',
        '--variant', variant,
        '--task-config', task_cfg_path,
        '--split', SPLIT,
        '--output-file', str(out_file),
    ]
    if BEAM_OVERRIDE is not None:
        cmd += ['--beam-override', str(BEAM_OVERRIDE)]
    if MAX_EXAMPLES is not None:
        cmd += ['--max-examples', str(MAX_EXAMPLES)]
    if variant == 'mole':
        cmd += ['--mole-config', mole_cfg_path,
                '--mole-router', 'mole_e2e_dart_webnlg/checkpoints/router_final.pt']
    t0 = time.time()
    env = os.environ | {'TOKENIZERS_PARALLELISM': 'false', 'TRANSFORMERS_VERBOSITY': 'error'}
    res = subprocess.run(cmd, env=env)
    elapsed = time.time() - t0
    if res.returncode != 0:
        raise RuntimeError(f'generation failed for ({variant}, {task_name})')
    return elapsed

# Outer tqdm gives a "cell 7/15 [hh:mm<rem]" summary across the matrix;
# the inner script's tqdm shows batch progress within each cell.
cells = [(v, mc, t, tc) for (v, mc) in VARIANTS for (t, tc) in TASKS]
total = len(cells)
outer = tqdm(cells, total=total, desc='matrix-gen', unit='cell')
for k, (variant, mole_cfg, task_name, task_cfg) in enumerate(outer, start=1):
    outer.set_postfix_str(f'{variant} on {task_name}')
    print(f'\n[gen {k}/{total}] === {variant} on {task_name} ===', flush=True)
    elapsed = gen_one(variant, mole_cfg, task_name, task_cfg)
    print(f'[gen {k}/{total}] -> done in {elapsed:.0f}s', flush=True)


## 7. Score All Cells

Same scorer for every (variant, task): sacrebleu BLEU + NLTK METEOR + pyter3 TER. Per-cell metric JSON lives next to the predictions JSONL.

In [ ]:
from tqdm.auto import tqdm

def score_one(variant, task_name, task_cfg_path):
    out_dir = EVAL_OUT_LOCAL / variant / task_name
    pred_file = out_dir / f'{SPLIT}_predictions.jsonl'
    metrics_file = out_dir / f'{SPLIT}_metrics.json'
    cmd = [
        'python', 'scripts/eval_matrix_evaluate.py',
        '--task-config', task_cfg_path,
        '--split', SPLIT,
        '--predictions-file', str(pred_file),
        '--output-file', str(metrics_file),
        '--variant', variant,
    ]
    res = subprocess.run(cmd)
    if res.returncode != 0:
        raise RuntimeError(f'eval failed for ({variant}, {task_name})')

cells = [(v, t, tc) for (v, _) in VARIANTS for (t, tc) in TASKS]
total = len(cells)
outer = tqdm(cells, total=total, desc='matrix-score', unit='cell')
for k, (variant, task_name, task_cfg) in enumerate(outer, start=1):
    outer.set_postfix_str(f'{variant} on {task_name}')
    print(f'\n[score {k}/{total}] === {variant} on {task_name} ===', flush=True)
    score_one(variant, task_name, task_cfg)


## 8. Aggregate The 5 × 3 Matrix

In [ ]:
import json
from pathlib import Path

def read_metrics(variant, task_name):
    path = EVAL_OUT_LOCAL / variant / task_name / f'{SPLIT}_metrics.json'
    return json.loads(path.read_text())['metrics']

table = {}
for variant, _ in VARIANTS:
    table[variant] = {}
    for task_name, _ in TASKS:
        m = read_metrics(variant, task_name)
        table[variant][task_name] = m

def fmt(metric_name, fmt_str='{:.2f}'):
    print(f'\n--- {metric_name.upper()} (rows = model, cols = task) ---')
    header = '{:>14}'.format('model') + ''.join('{:>10}'.format(t) for t,_ in TASKS)
    print(header)
    for variant, _ in VARIANTS:
        row = '{:>14}'.format(variant)
        for task_name, _ in TASKS:
            row += '{:>10}'.format(fmt_str.format(table[variant][task_name][metric_name]))
        print(row)

fmt('bleu')
fmt('meteor')
fmt('ter', '{:.3f}')

# Save a flat aggregate file for easy ingestion downstream.
agg_path = EVAL_OUT_LOCAL / f'{SPLIT}_matrix.json'
agg_path.write_text(json.dumps(table, indent=2, sort_keys=True))
print(f'\nwrote {agg_path}')

## 9. Backup To Drive

In [ ]:
backup(EVAL_OUT_LOCAL, dest_name='.')
backup('configs/mole_e2e_dart_webnlg.yaml', dest_name='mole_config.yaml')
backup('colab_eval_matrix.ipynb')

print('drive:', EVAL_OUT_DRIVE)
!find /content/drive/MyDrive/eval_matrix -maxdepth 4 -type f | sort | tail -30